# Opportunity Scout AI — Data Pipeline

**Architecture:**
```
User Input (industry, country, year)
        ↓
AggregationEngine
        ↓
Live Collectors:                    Fallback:
- GoogleTrendsCollector             VenturesCollector (ventures.csv)
- WorldBankCollector
- RedditCollector
- GDELTCollector          ← NEW
        ↓
Rate Limiter (per-collector)        ← NEW
        ↓
Signal Validation
        ↓
Normalization (stored min/max)
        ↓
Opportunity Score Engine
        ↓
Data Quality Scorer                 ← NEW
        ↓
Output JSON  ← mode: 'live' | 'historical'
              data_quality: 0.0-1.0 ← NEW
```

**Modes:**
- `Mode A (historical)` — batch-scores rows from ventures.csv (used by ML team for training)
- `Mode B (live)` — scores a new user query using live collectors + CSV fallback

#### Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import datetime
import time
import unittest
import threading
from abc import ABC, abstractmethod
from collections import deque


!pip install --upgrade google-cloud-bigquery
!pip install pytrends

Requirement already up-to-date: google-cloud-bigquery in c:\users\hp\anaconda3\envs\learn-env\lib\site-packages (3.30.0)


#### Load Dataset & Explore

In [2]:
df = pd.read_csv("../Data/ventures.csv")

print(df.columns.tolist())
print(df.head())
print(df.info())
print(df.describe())

['name', 'industry', 'category', 'founded_year', 'country', 'status', 'funding_total_usd', 'funding_rounds', 'opportunity_score', 'opportunity_category', 'trend_slope', 'news_volume', 'news_sentiment', 'reddit_density', 'gdp_growth', 'inflation', 'startup_density', 'log_funding']
                 name    industry               category  founded_year  \
0            #waywire        News  Media & Entertainment          2012   
1    Rock' Your Paper  Publishing  Media & Entertainment          2012   
2  -R- Ranch and Mine     Tourism   Travel & Hospitality          2014   
3    004 Technologies    Software  Technology & Software          2010   
4         1,2,3 Listo  E-Commerce    E-Commerce & Retail          2012   

         country     status  funding_total_usd  funding_rounds  \
0  United States   acquired            1750000               1   
1        Estonia  operating              40000               1   
2  United States  operating              60000               2   
3  United 

#### Schema & Feature Bounds

These bounds are derived from ventures.csv and saved to disk.
The ML team and API team will reference `feature_bounds.csv` for consistent normalization.

In [3]:
# Core signals the pipeline must produce
CORE_SCHEMA = {
    "trend_slope":      float,
    "news_volume":      float,
    "news_sentiment":   float,
    "reddit_density":   float,
    "gdp_growth":       float,
    "inflation":        float,
    "startup_density":  float,
}

# Features used in scoring + normalization
SCORE_FEATURES = [
    "trend_slope",
    "gdp_growth",
    "reddit_density",
    "news_sentiment",
    "news_volume",
    "startup_density",
    "log_funding",
    "inflation",
]

# Features the validator checks for in every output
REQUIRED_FEATURES = [
    "trend_slope",
    "gdp_growth",
    "reddit_density",
    "news_sentiment",
    "news_volume",
    "startup_density",
    "funding_total_usd",  # raw — log-transformed inside scorer
    "inflation",
]

# Which features are considered 'live' (sourced from live collectors, not CSV)
# Used by the data quality scorer to determine how fresh the signals are
LIVE_FEATURES = [
    "trend_slope",     # GoogleTrends
    "gdp_growth",      # WorldBank
    "inflation",       # WorldBank
    "reddit_density",  # Reddit
    "news_volume",     # GDELT
    "news_sentiment",  # GDELT
]

# Compute and save feature bounds from the dataset
bounds_list = []
for feature in SCORE_FEATURES:
    bounds_list.append({
        "feature": feature,
        "min":     df[feature].min(),
        "max":     df[feature].max(),
        "median":  df[feature].median(),
    })

feature_bounds_df = pd.DataFrame(bounds_list).set_index("feature")
feature_bounds_df.to_csv("feature_bounds.csv")
print("Feature bounds saved.")
feature_bounds_df

Feature bounds saved.


,min,max,median
feature,,,
trend_slope,-0.231172,0.364155,-0.055052
gdp_growth,-16.040020,14.519750,2.117830
reddit_density,0.000000,280.337400,0.702800
news_sentiment,-0.011100,6.592600,6.415900
news_volume,9041.000000,989564.000000,149705.000000
startup_density,0.003000,18.484600,5.092400
log_funding,0.000000,9.414973,5.989107
inflation,-4.447547,53.228698,2.138384


#### Logging & In-Memory Cache

In [4]:
# ── Logging ──────────────────────────────────────────────────────────────────

def log_event(level, message):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] [{level}] {message}")


# ── In-Memory Cache ───────────────────────────────────────────────────────────
# NOTE: For production, swap CACHE dict with Redis (see project plan Phase 1.2)

CACHE = {}

def get_cache_key(industry, country, year):
    return f"{industry}|{country}|{year}"

def get_cached(key):
    entry = CACHE.get(key)
    if entry:
        log_event("INFO", f"Cache hit for key: {key}")
    return entry

def set_cache(key, value):
    CACHE[key] = value
    log_event("INFO", f"Cached result for key: {key}")

def clear_cache():
    CACHE.clear()
    log_event("INFO", "Cache cleared")

#### Rate Limiter

A sliding-window rate limiter used to wrap every collector call.
Each collector gets its own `RateLimiter` instance with its own call budget.
This prevents hitting API rate limits and getting blocked.

| Collector       | Default limit       | Why                                       |
|-----------------|---------------------|-------------------------------------------|
| GoogleTrends    | 5 calls / 60 sec    | Unofficial API — very aggressive blocking |
| WorldBank       | 10 calls / 60 sec   | Free public API, generous but be polite   |
| Reddit          | 10 calls / 60 sec   | Public JSON, no auth = strict limits      |
| GDELT           | 5 calls / 60 sec    | Free API, but penalises rapid queries     |

In [5]:
class RateLimiter:
    """
    Sliding-window rate limiter.
    Tracks timestamps of recent calls in a deque.
    If the budget is exhausted, blocks (sleeps) until the oldest call
    falls outside the window.

    Usage:
        limiter = RateLimiter(max_calls=5, period_seconds=60)
        limiter.wait()   # call this before every API request
    """

    def __init__(self, max_calls: int, period_seconds: float):
        self.max_calls   = max_calls
        self.period      = period_seconds
        self._call_times = deque()
        self._lock       = threading.Lock()

    def wait(self):
        """Block if rate limit would be exceeded, then register this call."""
        with self._lock:
            now = time.monotonic()

            # Drop timestamps older than the window
            while self._call_times and now - self._call_times[0] > self.period:
                self._call_times.popleft()

            if len(self._call_times) >= self.max_calls:
                # Wait until oldest call expires from the window
                sleep_for = self.period - (now - self._call_times[0])
                if sleep_for > 0:
                    log_event("INFO", f"RateLimiter: sleeping {sleep_for:.1f}s to respect limit")
                    time.sleep(sleep_for)

                # Re-prune after sleep
                now = time.monotonic()
                while self._call_times and now - self._call_times[0] > self.period:
                    self._call_times.popleft()

            self._call_times.append(time.monotonic())

    def remaining(self) -> int:
        """How many calls are left in the current window."""
        now    = time.monotonic()
        active = sum(1 for t in self._call_times if now - t <= self.period)
        return max(0, self.max_calls - active)


# ── Shared limiter instances (one per collector) ──────────────────────────────
RATE_LIMITERS = {
    "GoogleTrends": RateLimiter(max_calls=5,  period_seconds=60),
    "WorldBank":    RateLimiter(max_calls=10, period_seconds=60),
    "Reddit":       RateLimiter(max_calls=10, period_seconds=60),
    "GDELT":        RateLimiter(max_calls=5,  period_seconds=60),
}

#### Signal Validation

In [6]:
def validate_signals(signals):
    """
    Checks all REQUIRED_FEATURES are present, numeric, and not NaN.
    Returns:
        validated (dict) — cleaned signals
        issues    (dict) — feature -> issue description
    """
    validated = {}
    issues    = {}

    for feature in REQUIRED_FEATURES:

        if feature not in signals:
            validated[feature] = np.nan
            issues[feature]    = "missing_key"
            continue

        value = signals[feature]

        try:
            validated[feature] = float(value)
            if np.isnan(validated[feature]):
                issues[feature] = "nan_value"
        except (TypeError, ValueError):
            validated[feature] = np.nan
            issues[feature]    = "type_error"

    # Pass-through any extra signals
    for k, v in signals.items():
        if k not in validated:
            try:
                validated[k] = float(v)
            except (TypeError, ValueError):
                pass

    if issues:
        log_event("WARN", f"Signal issues detected: {issues}")

    return validated, issues

#### Data Quality Scorer

After validation, we score the *quality* of the signals collected.
Quality is a float from 0.0 to 1.0 and is included in every output JSON.

**How it works:**
- Each of the 6 `LIVE_FEATURES` is worth a fraction of the total score.
- If the signal came from a live collector → full credit (1.0)
- If it came from the CSV fallback → half credit (0.5) — it's real but historical
- If it is NaN or missing entirely → no credit (0.0)

**Quality label thresholds:**
- `High`   ≥ 0.75  — most signals are from live APIs, scores are trustworthy
- `Medium` ≥ 0.50  — mix of live and fallback, treat score with some caution
- `Low`    < 0.50  — mostly fallback or imputed, low confidence in the score

The ML team can use `data_quality` as a sample weight during training.
The frontend should display the `quality_label` badge next to the score.

In [7]:
def score_data_quality(validated_signals: dict, live_signals: dict) -> dict:
    """
    Measures how much of the data came from live sources vs fallback vs missing.

    Args:
        validated_signals : output of validate_signals() — all signals present
        live_signals      : signals dict assembled ONLY from live collectors
                            (before fallback was applied)

    Returns dict with:
        data_quality       : float 0.0 - 1.0
        quality_label      : 'High' | 'Medium' | 'Low'
        quality_breakdown  : per-feature source label
    """
    total_features = len(LIVE_FEATURES)
    score_sum      = 0.0
    breakdown      = {}

    for feature in LIVE_FEATURES:
        value = validated_signals.get(feature)

        # Missing or NaN
        if value is None or (isinstance(value, float) and np.isnan(value)):
            breakdown[feature] = "missing"
            score_sum += 0.0

        # Came from live collector
        elif feature in live_signals and live_signals[feature] is not None:
            breakdown[feature] = "live"
            score_sum += 1.0

        # Came from CSV fallback
        else:
            breakdown[feature] = "fallback"
            score_sum += 0.5

    quality_score = round(score_sum / total_features, 4)

    if quality_score >= 0.75:
        label = "High"
    elif quality_score >= 0.50:
        label = "Medium"
    else:
        label = "Low"

    log_event("INFO", f"Data quality: {quality_score} ({label}) | breakdown: {breakdown}")

    return {
        "data_quality":      quality_score,
        "quality_label":     label,
        "quality_breakdown": breakdown,
    }

#### Normalization Helpers

In [8]:
def normalize(value, feature_name):
    """Scale value to 0-100 using stored min/max bounds."""
    min_val = feature_bounds_df.loc[feature_name, "min"]
    max_val = feature_bounds_df.loc[feature_name, "max"]
    if max_val == min_val:
        return 50.0
    return float(np.clip(((value - min_val) / (max_val - min_val)) * 100, 0, 100))


def normalize_inverted(value, feature_name):
    """Like normalize() but higher raw value → lower score (used for inflation)."""
    return 100 - normalize(value, feature_name)

#### Opportunity Scorer

In [9]:
WEIGHTS = {
    "trend_slope":     0.20,
    "gdp_growth":      0.15,
    "reddit_density":  0.10,
    "news_sentiment":  0.10,
    "news_volume":     0.10,
    "startup_density": 0.10,
    "log_funding":     0.15,
    "inflation":       0.10,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-6, "Weights must sum to 1.0"


def compute_opportunity_score(signals):
    """
    Input : validated signals dict (from validate_signals)
    Output: float score 0-100

    BUG FIX: imputation now covers ALL eight scored signals, not just four.
    Previously trend_slope, news_sentiment and funding_total_usd were not imputed,
    so a NaN in any of those propagated through normalize() and poisoned the entire
    weighted sum with NaN. A final NaN guard is also applied before returning.
    """
    s = dict(signals)

    # Impute ALL missing/NaN scored signals with their dataset medians.
    # Previously only gdp_growth, inflation, news_volume, startup_density were covered.
    # trend_slope, news_sentiment and funding_total_usd were silently left as NaN.
    for feature in ["trend_slope", "gdp_growth", "inflation",
                    "news_volume", "news_sentiment", "startup_density",
                    "reddit_density"]:
        val = s.get(feature)
        if val is None or (isinstance(val, float) and (np.isnan(val) or np.isinf(val))):
            median = (
                float(feature_bounds_df.loc[feature, "median"])
                if feature in feature_bounds_df.index else 0.0
            )
            s[feature] = median
            log_event("WARN", f"Imputed missing '{feature}' with median ({median})")

    # funding_total_usd: impute with exp(log_funding_median) - 1 so log transform stays valid
    raw_funding = s.get("funding_total_usd")
    if raw_funding is None or (isinstance(raw_funding, float) and (np.isnan(raw_funding) or np.isinf(raw_funding))):
        log_funding_median = float(feature_bounds_df.loc["log_funding", "median"]) \
            if "log_funding" in feature_bounds_df.index else 0.0
        raw_funding = float(np.expm1(log_funding_median))
        s["funding_total_usd"] = raw_funding
        log_event("WARN", f"Imputed missing 'funding_total_usd' from log_funding median")

    # Log-transform funding
    s["log_funding"] = float(np.log1p(max(0, raw_funding)))

    # Normalize each signal (all NaNs now imputed so normalize() is safe)
    normalized = {
        "trend_slope":     normalize(s["trend_slope"],       "trend_slope"),
        "gdp_growth":      normalize(s["gdp_growth"],        "gdp_growth"),
        "reddit_density":  normalize(s["reddit_density"],    "reddit_density"),
        "news_sentiment":  normalize(s["news_sentiment"],    "news_sentiment"),
        "news_volume":     normalize(s["news_volume"],       "news_volume"),
        "startup_density": normalize(s["startup_density"],   "startup_density"),
        "log_funding":     normalize(s["log_funding"],       "log_funding"),
        "inflation":       normalize_inverted(s["inflation"], "inflation"),
    }

    score = sum(WEIGHTS[k] * normalized[k] for k in WEIGHTS)

    # Final NaN guard: if score is still NaN (should not happen after imputation),
    # return the neutral mid-point score of 50.0 rather than NaN.
    if np.isnan(score) or np.isinf(score):
        log_event("WARN", "Score was NaN/Inf after normalization — returning 50.0 (neutral)")
        return 50.0

    return round(float(score), 4)


#### Base Collector

In [10]:
class BaseCollector(ABC):
    """
    Every collector must implement collect().
    Returns a dict of signal_name → float value.
    On failure, returns an empty dict (fallback handled by AggregationEngine).
    """

    @abstractmethod
    def collect(self, industry: str, country: str, year: int) -> dict:
        pass

#### VenturesCollector (CSV Fallback)

In [11]:
class VenturesCollector(BaseCollector):
    """
    Reads from ventures.csv.
    Fuzzy-matches industry and supports year-range fallback.
    """

    def __init__(self, filepath):
        self.df = pd.read_csv(filepath)
        log_event("INFO", f"VenturesCollector loaded {len(self.df)} rows from {filepath}")

    def collect(self, industry: str, country: str, year: int) -> dict:
        # Exact match first
        filtered = self.df[
            (self.df["industry"].str.lower() == industry.lower()) &
            (self.df["country"].str.lower()  == country.lower())  &
            (self.df["founded_year"]          == year)
        ]

        # Year-range fallback (+-3 years)
        if filtered.empty:
            log_event("WARN", f"No exact match for ({industry}, {country}, {year}). Trying +-3yr range.")
            filtered = self.df[
                (self.df["industry"].str.lower() == industry.lower()) &
                (self.df["country"].str.lower()  == country.lower())  &
                (self.df["founded_year"].between(year - 3, year + 3))
            ]

        if filtered.empty:
            log_event("WARN", f"VenturesCollector: no data found for ({industry}, {country}, {year})")
            return {}

        row = filtered[[
            "trend_slope", "gdp_growth", "reddit_density",
            "news_sentiment", "news_volume", "startup_density",
            "funding_total_usd", "inflation"
        ]].median()

        return row.to_dict()

    def get_all_rows(self) -> pd.DataFrame:
        """Return full dataset for Mode A (historical batch scoring)."""
        return self.df.copy()

#### GoogleTrendsCollector

In [12]:
# Install if needed:  pip install pytrends

class GoogleTrendsCollector(BaseCollector):
    """
    Fetches trend_slope from Google Trends via pytrends.
    Uses RATE_LIMITERS['GoogleTrends'] before every API call.
    """

    COUNTRY_CODE_MAP = {
        "united states": "US", "kenya": "KE", "united kingdom": "GB",
        "india": "IN", "germany": "DE", "france": "FR", "canada": "CA",
        "australia": "AU", "nigeria": "NG", "south africa": "ZA",
    }

    def collect(self, industry: str, country: str, year: int) -> dict:
        try:
            from pytrends.request import TrendReq

            RATE_LIMITERS["GoogleTrends"].wait()   # rate limit enforced here

            geo       = self.COUNTRY_CODE_MAP.get(country.lower(), "")
            pytrends  = TrendReq(hl="en-US", tz=360, timeout=(10, 25))
            timeframe = f"{year}-01-01 {year}-12-31"
            pytrends.build_payload([industry], cat=0, timeframe=timeframe, geo=geo)

            data = pytrends.interest_over_time()

            if data.empty or industry not in data.columns:
                log_event("WARN", f"GoogleTrends: no data for '{industry}'")
                return {}

            series = data[industry].values.astype(float)
            x      = np.arange(len(series))
            if len(series) < 2:
                return {}

            slope = float(np.polyfit(x, series, 1)[0]) / 100.0
            log_event("INFO", f"GoogleTrends: trend_slope={slope:.4f}")
            return {"trend_slope": slope}

        except Exception as e:
            log_event("ERROR", f"GoogleTrendsCollector failed: {str(e)}")
            return {}

#### WorldBankCollector

In [13]:
class WorldBankCollector(BaseCollector):
    """
    Fetches gdp_growth and inflation from the World Bank Open Data API.
    No API key required. Uses RATE_LIMITERS['WorldBank'].
    """

    BASE_URL = "https://api.worldbank.org/v2/country/{iso}/indicator/{indicator}"

    COUNTRY_ISO_MAP = {
        "united states": "US", "kenya": "KE", "united kingdom": "GB",
        "india": "IN", "germany": "DE", "france": "FR", "canada": "CA",
        "australia": "AU", "nigeria": "NG", "south africa": "ZA",
        "china": "CN", "brazil": "BR", "chile": "CL", "estonia": "EE",
    }

    def _fetch_indicator(self, iso, indicator, year):
        RATE_LIMITERS["WorldBank"].wait()   # rate limit enforced here
        url    = self.BASE_URL.format(iso=iso, indicator=indicator)
        params = {"date": str(year), "format": "json", "per_page": 1}
        resp   = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        payload = resp.json()
        if len(payload) < 2 or not payload[1]:
            return None
        value = payload[1][0].get("value")
        return float(value) if value is not None else None

    def collect(self, industry: str, country: str, year: int) -> dict:
        iso = self.COUNTRY_ISO_MAP.get(country.lower())
        if not iso:
            log_event("WARN", f"WorldBankCollector: no ISO code for '{country}'")
            return {}

        result = {}
        try:
            gdp = self._fetch_indicator(iso, "NY.GDP.MKTP.KD.ZG", year)
            if gdp is not None:
                result["gdp_growth"] = gdp
                log_event("INFO", f"WorldBank: gdp_growth={gdp}")
        except Exception as e:
            log_event("ERROR", f"WorldBank GDP failed: {str(e)}")

        try:
            inf = self._fetch_indicator(iso, "FP.CPI.TOTL.ZG", year)
            if inf is not None:
                result["inflation"] = inf
                log_event("INFO", f"WorldBank: inflation={inf}")
        except Exception as e:
            log_event("ERROR", f"WorldBank Inflation failed: {str(e)}")

        return result

#### RedditCollector

In [14]:
class RedditCollector(BaseCollector):
    """
    Fetches reddit_density using the Pushshift/PullPush archive API.
    Mirrors the approach in red.py: keyword-based, year-scoped, daily density.
    Uses RATE_LIMITERS['Reddit'] before every request.

    Why PullPush instead of Reddit's live API:
    - Reddit's public /search.json ignores the query and returns trending posts,
      producing the same 100-post count regardless of keyword — fake data.
    - PullPush indexes historical Reddit submissions by keyword and timestamp,
      returning genuine keyword-specific counts for any year.
    """

    SEARCH_URL = "https://api.pullpush.io/reddit/search/submission/"
    HEADERS    = {"User-Agent": "OpportunityScoutAI/1.0"}

    # Industry → keyword list, mirroring red.py INDUSTRY_KEYWORDS
    INDUSTRY_KEYWORDS = {
        'E-Commerce & Retail':            ['ecommerce', 'online shopping', 'retail technology'],
        'Technology & Software':          ['software', 'saas', 'cloud computing'],
        'Healthcare & Life Sciences':     ['digital health', 'telemedicine', 'healthcare technology'],
        'Finance & Fintech':              ['fintech', 'digital banking', 'payment technology'],
        'Finance Technology':             ['fintech', 'digital banking', 'payment technology'],
        'Media & Entertainment':          ['streaming services', 'digital media', 'content creation'],
        'Education & Training':           ['edtech', 'online learning', 'educational technology'],
        'Real Estate & Construction':     ['proptech', 'real estate technology', 'construction technology'],
        'Travel & Hospitality':           ['travel technology', 'online booking', 'hotel technology'],
        'Food & Beverage':                ['food delivery', 'food technology', 'restaurant technology'],
        'Transportation & Logistics':     ['logistics technology', 'supply chain', 'delivery services'],
        'Marketing & Advertising':        ['digital marketing', 'martech', 'advertising technology'],
        'Internet & Web Services':        ['web services', 'internet technology', 'web applications'],
        'Data & Analytics':               ['data analytics', 'big data', 'business intelligence'],
        'Gaming':                         ['video games', 'gaming', 'esports'],
        'Social & Community':             ['social media', 'social networking', 'online community'],
        'Consumer & Lifestyle':           ['consumer technology', 'lifestyle products', 'personal technology'],
        'Manufacturing & Industrial':     ['industrial automation', 'manufacturing technology', 'industry 4.0'],
        'Sports & Recreation':            ['fitness technology', 'sports technology', 'wellness'],
        'Communications & Telecom':       ['telecommunications', '5g technology', 'telecom'],
        'Energy & Clean Tech':            ['clean energy', 'renewable energy', 'cleantech'],
        'Enterprise & Business Services': ['business software', 'enterprise solutions', 'b2b software'],
        'Security & Privacy':             ['cybersecurity', 'information security', 'data privacy'],
        'AI & Machine Learning':          ['artificial intelligence', 'machine learning', 'deep learning'],
        'Customer Service & CRM':         ['crm software', 'customer service', 'customer experience'],
        'HR & Recruiting':                ['recruitment technology', 'hr software', 'talent management'],
        'Government & Politics':          ['govtech', 'government technology', 'civic technology'],
        'Software':                       ['software', 'saas', 'cloud computing'],
        'News':                           ['news media', 'digital news', 'online journalism'],
        'Publishing':                     ['digital publishing', 'ebooks', 'online publishing'],
        'Tourism':                        ['travel technology', 'tourism technology', 'online booking'],
    }

    def _keyword_density(self, keyword: str, year: int) -> float:
        """Fetch posts for one keyword in one year and return daily density."""
        # epoch timestamps for Jan 1 and Dec 31 of target year
        start_ts = int(time.mktime(time.strptime(f"{year}-01-01", "%Y-%m-%d")))
        end_ts   = int(time.mktime(time.strptime(f"{year}-12-31", "%Y-%m-%d")))

        RATE_LIMITERS["Reddit"].wait()
        params = {
            "q":     keyword,
            "after": start_ts,
            "before": end_ts,
            "size":  100,
            "sort":  "asc",
        }
        try:
            resp = requests.get(self.SEARCH_URL, params=params,
                                headers=self.HEADERS, timeout=15)
            if resp.status_code != 200:
                log_event("WARN", f"PullPush {resp.status_code} for '{keyword}'")
                return 0.0
            data = resp.json().get("data", [])
            if len(data) < 2:
                return 0.0
            # compute daily density from first to last post timestamp
            # PullPush sometimes returns created_utc as a string — cast to int defensively
            days = (int(data[-1]["created_utc"]) - int(data[0]["created_utc"])) / 86400
            return round(len(data) / days, 4) if days > 0 else 0.0
        except Exception as e:
            log_event("ERROR", f"PullPush keyword '{keyword}': {str(e)}")
            return 0.0

    def collect(self, industry: str, country: str, year: int) -> dict:
        try:
            # Map industry name to keyword list
            keywords = self.INDUSTRY_KEYWORDS.get(industry)
            if not keywords:
                # Generic fallback: use industry name itself as a single keyword
                keywords = [industry.lower()]
                log_event("WARN", f"RedditCollector: no keyword map for '{industry}', using name as query")

            densities = [self._keyword_density(kw, year) for kw in keywords]
            avg_density = round(sum(densities) / len(densities), 4)

            log_event("INFO", f"Reddit (PullPush): '{industry}' {year} -> density={avg_density}")
            return {"reddit_density": avg_density}

        except Exception as e:
            log_event("ERROR", f"RedditCollector failed: {str(e)}")
            return {}


#### GDELTCollector

Uses **Google BigQuery** to query the GDELT dataset — the same approach used in `gdelt_test.py`
which was used to build the original ventures.csv signals.

**Why BigQuery instead of the GDELT Doc API:**
- The free GDELT Doc 2.0 REST API timed out on every live run (confirmed in notebook logs).
- BigQuery is reliable, fast, and uses the same service account key (`strong_gdelt.json`) you already have.
- The `_bq_cache` dict ensures the same industry+year is only queried once per session.

**Setup:** Place `strong_gdelt.json` in the same folder as the notebook before running.

**Signals returned:**
- `news_volume` — article count for the industry's GDELT theme code in the target year
- `news_sentiment` — GDELT AvgTone mapped from (-10,+10) to (0, 6.6) to match ventures.csv scale

**Year routing:**
- 2015+ uses `gdelt-bq.gdeltv2.gkg_partitioned` (same as gdelt_test.py Batch 1)
- 2005-2014 uses `gdelt-bq.full.events` filtered to business actors (same as Batch 2)

In [15]:
class GDELTCollector(BaseCollector):
    """
    Fetches news_volume and news_sentiment via Google BigQuery (GDELT dataset).
    Mirrors the approach in gdelt_test.py: service account auth + SQL on gdelt-bq.

    Why BigQuery instead of the GDELT Doc 2.0 REST API:
    - The free GDELT Doc API times out consistently (confirmed in live run logs).
    - BigQuery gives reliable, fast access to the same underlying data.
    - The service account key (strong_gdelt.json) authenticates automatically.

    Results are cached in _bq_cache (dict) so the same industry+year is only
    queried once per notebook session regardless of how many times collect() is called.

    news_sentiment: GDELT AvgTone mapped from (-10,+10) -> (0,6.6) to match
                    ventures.csv scale.
    """

    PROJECT_ID    = "strong-augury-487515-u7"
    CREDENTIALS   = "../python_files/strong_gdelt.json"

    # Same industry->theme-code map as gdelt_test.py
    INDUSTRY_MAP = {
        'Media & Entertainment':          'GENERAL_ENTERTAINMENT',
        'Travel & Hospitality':           'TOURISM',
        'Technology & Software':          'TECHNOLOGY',
        'E-Commerce & Retail':            'ECONOMY_ECOMMERCE',
        'Real Estate & Construction':     'ECONOMY_REALESTATE',
        'Education & Training':           'EDUCATION',
        'Internet & Web Services':        'INTERNET',
        'Food & Beverage':                'FOOD_SECURITY',
        'Healthcare & Life Sciences':     'HEALTH_SERVICES',
        'Data & Analytics':               'COMPUTING_DATA_MINING',
        'Consumer & Lifestyle':           'SOCIETY_LIFESTYLE',
        'Transportation & Logistics':     'TRANSPORTATION',
        'Sports & Recreation':            'SPORTS',
        'Manufacturing & Industrial':     'MANUFACTURING',
        'Marketing & Advertising':        'ADVERTISING',
        'Gaming':                         'GAMES',
        'Finance & Fintech':              'ECONOMY_FINTECH',
        'Finance Technology':             'ECONOMY_FINTECH',
        'Communications & Telecom':       'TELECOMMUNICATIONS',
        'Social & Community':             'SOCIETY',
        'Energy & Clean Tech':            'ENERGY_RENEWABLE',
        'Enterprise & Business Services': 'ECONOMY_BUSINESS_SERVICES',
        'AI & Machine Learning':          'COMPUTING_ARTIFICIAL_INTELLIGENCE',
        'Customer Service & CRM':         'CUSTOMER_SERVICE',
        'HR & Recruiting':                'LABOR_RECRUITMENT',
        'Security & Privacy':             'SECURITY_SERVICES',
        'Government & Politics':          'GOVERNMENT',
        'Software':                       'TECHNOLOGY',
        'News':                           'GENERAL_ENTERTAINMENT',
        'Publishing':                     'GENERAL_ENTERTAINMENT',
        'Tourism':                        'TOURISM',
    }

    # In-memory cache so the same BQ query never runs twice in a session
    _bq_cache: dict = {}

    def _get_client(self):
        """
        Lazy-init BigQuery client using the service account JSON.
        Searches the notebook folder and common parent directories automatically.
        Set CREDENTIALS to a full absolute path to skip the search.
        """
        import os
        from google.cloud import bigquery

        creds_path = self.CREDENTIALS

        # If it is not a full path and the file is not found in cwd,
        # walk up two directory levels to find it (handles running from subfolders)
        if not os.path.isabs(creds_path) and not os.path.isfile(creds_path):
            for parent in [
                os.path.dirname(os.path.abspath(".")),          # one level up
                os.path.dirname(os.path.dirname(os.path.abspath("."))),  # two levels up
                os.path.abspath("."),                            # cwd (already tried)
            ]:
                candidate = os.path.join(parent, os.path.basename(creds_path))
                if os.path.isfile(candidate):
                    creds_path = candidate
                    log_event("INFO", f"GDELT: found credentials at {creds_path}")
                    break
            else:
                raise FileNotFoundError(
                    f"Credentials file '{self.CREDENTIALS}' not found.\n"
                    f"Place strong_gdelt.json in the same folder as the notebook, "
                    f"or set GDELTCollector.CREDENTIALS to its full absolute path."
                )

        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = creds_path
        return bigquery.Client(project=self.PROJECT_ID, location="US")

    def _query_bq(self, theme_code: str, year: int) -> dict:
        """Run the BigQuery SQL and return {news_volume, news_sentiment}."""
        from google.cloud import bigquery
        client = self._get_client()

        if year >= 2015:
            # Partitioned table — efficient for 2015+
            sql = f"""
                SELECT
                    COUNT(*) as ArticleVolume,
                    ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64)), 4) as AvgSentiment
                FROM `gdelt-bq.gdeltv2.gkg_partitioned`
                WHERE _PARTITIONDATE BETWEEN '{year}-01-01' AND '{year}-12-31'
                  AND themes LIKE '%{theme_code}%'
            """
        else:
            # Historical table for 2005-2014
            sql = f"""
                SELECT
                    COUNT(*) as ArticleVolume,
                    ROUND(AVG(AvgTone), 4) as AvgSentiment
                FROM `gdelt-bq.full.events`
                WHERE Year = {year}
                  AND (Actor1Type1Code = 'BUS' OR Actor2Type1Code = 'BUS')
            """

        job    = client.query(sql)
        rows   = list(job.result())
        if not rows or rows[0].ArticleVolume == 0:
            return {}

        raw_volume    = int(rows[0].ArticleVolume)
        raw_sentiment = rows[0].AvgSentiment  # GDELT AvgTone: roughly -10 to +10

        # Map AvgTone (-10,+10) -> (0, 6.6) to match ventures.csv news_sentiment scale
        if raw_sentiment is not None:
            mapped_sentiment = round(float(np.clip((raw_sentiment + 10) * (6.6 / 20.0), 0, 6.6)), 4)
        else:
            mapped_sentiment = None

        result = {"news_volume": float(raw_volume)}
        if mapped_sentiment is not None:
            result["news_sentiment"] = mapped_sentiment
        return result

    def collect(self, industry: str, country: str, year: int) -> dict:
        try:
            theme_code = self.INDUSTRY_MAP.get(industry)
            if not theme_code:
                log_event("WARN", f"GDELTCollector: no theme code for '{industry}', skipping")
                return {}

            cache_key = f"{theme_code}|{year}"
            if cache_key in self._bq_cache:
                log_event("INFO", f"GDELT BQ cache hit: {cache_key}")
                return self._bq_cache[cache_key]

            RATE_LIMITERS["GDELT"].wait()
            result = self._query_bq(theme_code, year)

            if result:
                log_event("INFO", f"GDELT BQ: '{industry}' {year} -> volume={result.get('news_volume')}, "
                          f"sentiment={result.get('news_sentiment')}")
            else:
                log_event("WARN", f"GDELT BQ: no results for '{industry}' {year}")

            self._bq_cache[cache_key] = result
            return result

        except Exception as e:
            log_event("ERROR", f"GDELTCollector failed: {str(e)}")
            return {}


####  AggregationEngine

- `run_live()` — Mode B. Calls live collectors (rate-limited), falls back to CSV, scores, quality-checks.
- `run_historical()` — Mode A. Batch-scores all rows from ventures.csv.

In [16]:
class AggregationEngine:

    def __init__(self, ventures_path: str):
        self.ventures_collector = VenturesCollector(ventures_path)
        self.live_collectors = [
            GoogleTrendsCollector(),
            WorldBankCollector(),
            RedditCollector(),
            GDELTCollector(),
        ]
        log_event("INFO", "AggregationEngine initialized")

    # MODE B — Live scoring (web app / API)

    def run_live(self, industry: str, country: str, year: int) -> dict:
        """
        Full live pipeline:
          1. Check cache
          2. Collect from live APIs (rate-limited)
          3. Fill gaps from ventures.csv fallback
          4. Validate signals
          5. Score opportunity
          6. Score data quality
          7. Return structured JSON

        Output fields:
          status, mode, input, score, data_quality, quality_label,
          quality_breakdown, signals, issues
        """
        cache_key = get_cache_key(industry, country, year)
        cached    = get_cached(cache_key)
        if cached:
            return cached

        # Step 1: Collect from live sources
        live_signals = {}
        for collector in self.live_collectors:
            name = collector.__class__.__name__
            try:
                data = collector.collect(industry, country, year)
                live_signals.update(data)
                log_event("INFO", f"{name} returned signals: {list(data.keys())}")
            except Exception as e:
                log_event("ERROR", f"{name} crashed unexpectedly: {str(e)}")

        # Step 2: Fill gaps with historical CSV fallback
        merged_signals = dict(live_signals)
        historical     = self.ventures_collector.collect(industry, country, year) or {}

        fallback_used = []
        for feature in REQUIRED_FEATURES:
            if feature not in merged_signals or merged_signals[feature] is None:
                fallback_val = historical.get(feature)
                if fallback_val is not None:
                    merged_signals[feature] = fallback_val
                    fallback_used.append(feature)

        if fallback_used:
            log_event("INFO", f"Fallback used for: {fallback_used}")

        # Step 3: Validate
        validated, issues = validate_signals(merged_signals)

        # Step 4: Score opportunity
        score = compute_opportunity_score(validated)

        # Step 5: Score data quality
        quality_info = score_data_quality(validated, live_signals)

        # Step 6: Build output
        result = {
            "status":            "success",
            "mode":              "live",
            "input": {
                "industry":  industry,
                "country":   country,
                "year":      year,
            },
            "score":             score,
            "data_quality":      quality_info["data_quality"],
            "quality_label":     quality_info["quality_label"],
            "quality_breakdown": quality_info["quality_breakdown"],
            "signals":           validated,
            "issues":            issues,
        }

        set_cache(cache_key, result)
        return result

    # MODE A — Historical batch scoring (ML team training data)

    def run_historical(self, sample_n: int = None) -> pd.DataFrame:
        """
        Batch-scores all (or sample_n) rows from ventures.csv.
        Returns DataFrame with original columns + 'pipeline_score' + 'data_quality'.
        This is what the ML team uses to train the scoring model.
        """
        source_df = self.ventures_collector.get_all_rows()

        if sample_n:
            source_df = source_df.sample(n=min(sample_n, len(source_df)), random_state=42)

        log_event("INFO", f"run_historical: scoring {len(source_df)} rows...")
        scores  = []
        quality = []

        for _, row in source_df.iterrows():
            signals = {
                "trend_slope":       row.get("trend_slope"),
                "gdp_growth":        row.get("gdp_growth"),
                "reddit_density":    row.get("reddit_density"),
                "news_sentiment":    row.get("news_sentiment"),
                "news_volume":       row.get("news_volume"),
                "startup_density":   row.get("startup_density"),
                "funding_total_usd": row.get("funding_total_usd"),
                "inflation":         row.get("inflation"),
            }
            validated, _  = validate_signals(signals)
            score         = compute_opportunity_score(validated)
            q_info        = score_data_quality(validated, live_signals={})

            scores.append(score)
            quality.append(q_info["data_quality"])

        source_df                   = source_df.copy()
        source_df["pipeline_score"] = scores
        source_df["data_quality"]   = quality

        log_event("INFO", f"run_historical complete. Mean pipeline_score={np.mean(scores):.2f}")
        return source_df

#### Unit Tests

In [17]:
class TestPipeline(unittest.TestCase):

    def setUp(self):
        self.ventures_path = "../Data/ventures.csv"
        self.engine = AggregationEngine(self.ventures_path)

    def _full_signals(self):
        return {
            "trend_slope": 0.1, "gdp_growth": 2.5, "reddit_density": 5.0,
            "news_sentiment": 6.0, "news_volume": 150000.0,
            "startup_density": 4.0, "funding_total_usd": 500000.0, "inflation": 2.0
        }

    def test_validate_signals_valid(self):
        validated, issues = validate_signals(self._full_signals())
        self.assertEqual(issues, {})
        self.assertIn("trend_slope", validated)

    def test_validate_signals_missing_key(self):
        _, issues = validate_signals({"trend_slope": 0.1})
        self.assertIn("gdp_growth", issues)
        self.assertEqual(issues["gdp_growth"], "missing_key")

    def test_validate_signals_nan(self):
        sigs = self._full_signals()
        sigs["trend_slope"] = float("nan")
        _, issues = validate_signals(sigs)
        self.assertEqual(issues["trend_slope"], "nan_value")

    def test_scorer_range(self):
        validated, _ = validate_signals(self._full_signals())
        score = compute_opportunity_score(validated)
        self.assertGreaterEqual(score, 0)
        self.assertLessEqual(score, 100)

    def test_ventures_no_match(self):
        collector = VenturesCollector(self.ventures_path)
        result = collector.collect("NonexistentIndustryXYZ", "Mars", 1800)
        self.assertEqual(result, {})

    def test_live_mode_field(self):
        result = self.engine.run_live("Finance Technology", "United States", 2010)
        self.assertEqual(result["mode"], "live")

    def test_historical_returns_dataframe(self):
        df_result = self.engine.run_historical(sample_n=10)
        self.assertIsInstance(df_result, pd.DataFrame)
        self.assertIn("pipeline_score", df_result.columns)
        self.assertEqual(len(df_result), 10)

    def test_data_quality_all_live(self):
        sigs = self._full_signals()
        validated, _ = validate_signals(sigs)
        q = score_data_quality(validated, live_signals=validated)
        self.assertEqual(q["quality_label"], "High")
        self.assertGreaterEqual(q["data_quality"], 0.75)

    def test_data_quality_all_fallback(self):
        sigs = self._full_signals()
        validated, _ = validate_signals(sigs)
        q = score_data_quality(validated, live_signals={})
        self.assertEqual(q["quality_label"], "Medium")

    def test_rate_limiter_does_not_block_short_burst(self):
        limiter = RateLimiter(max_calls=5, period_seconds=60)
        start = time.monotonic()
        for _ in range(5):
            limiter.wait()
        elapsed = time.monotonic() - start
        self.assertLess(elapsed, 1.0)

    def test_output_has_quality_fields(self):
        result = self.engine.run_live("Finance Technology", "United States", 2010)
        self.assertIn("data_quality",      result)
        self.assertIn("quality_label",     result)
        self.assertIn("quality_breakdown", result)


suite  = unittest.TestLoader().loadTestsFromTestCase(TestPipeline)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

test_data_quality_all_fallback (__main__.TestPipeline) ... ok
test_data_quality_all_live (__main__.TestPipeline) ... ok
test_historical_returns_dataframe (__main__.TestPipeline) ... 

[2026-02-19 19:05:55] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-19 19:05:55] [INFO] AggregationEngine initialized
[2026-02-19 19:05:55] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-19 19:05:55] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-19 19:05:55] [INFO] AggregationEngine initialized
[2026-02-19 19:05:55] [INFO] Data quality: 1.0 (High) | breakdown: {'trend_slope': 'live', 'gdp_growth': 'live', 'inflation': 'live', 'reddit_density': 'live', 'news_volume': 'live', 'news_sentiment': 'live'}


ok
test_live_mode_field (__main__.TestPipeline) ... 

[2026-02-19 19:05:55] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-19 19:05:55] [INFO] AggregationEngine initialized
[2026-02-19 19:05:55] [INFO] run_historical: scoring 10 rows...
[2026-02-19 19:05:55] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-19 19:05:55] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-19 19:05:55] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-19 19:05:55] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slop

C:\Users\HP\anaconda3\envs\learn-env\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.8.5). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\HP\anaconda3\envs\learn-env\lib\site-packages\google\auth\__init__.py:52: FutureWarning: You are using a Python version 3.8 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.8"), FutureWarning)
C:\Users\HP\anaconda3\envs\learn-env\lib\site-packages\google\oauth2\__init__.py:38: FutureWarning: You are using a Python version 3.8 past its end of life. Google will update google-auth with c

...

[2026-02-19 19:06:08] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-19 19:06:08] [INFO] AggregationEngine initialized
[2026-02-19 19:06:08] [WARN] Signal issues detected: {'gdp_growth': 'missing_key', 'reddit_density': 'missing_key', 'news_sentiment': 'missing_key', 'news_volume': 'missing_key', 'startup_density': 'missing_key', 'funding_total_usd': 'missing_key', 'inflation': 'missing_key'}
[2026-02-19 19:06:08] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-19 19:06:08] [INFO] AggregationEngine initialized
[2026-02-19 19:06:08] [WARN] Signal issues detected: {'trend_slope': 'nan_value'}


ok
test_ventures_no_match (__main__.TestPipeline) ... 

[2026-02-19 19:06:08] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-19 19:06:08] [INFO] AggregationEngine initialized
[2026-02-19 19:06:08] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-19 19:06:08] [INFO] AggregationEngine initialized
[2026-02-19 19:06:08] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-19 19:06:08] [WARN] No exact match for (NonexistentIndustryXYZ, Mars, 1800). Trying +-3yr range.
[2026-02-19 19:06:08] [WARN] VenturesCollector: no data found for (NonexistentIndustryXYZ, Mars, 1800)


ok

----------------------------------------------------------------------
Ran 11 tests in 13.636s

OK


<unittest.runner.TextTestResult run=11 errors=0 failures=0>

#### Live Test

Tests the full Mode B pipeline end-to-end with real API calls.
Check `quality_breakdown` to see exactly which signals came from live collectors
vs the CSV fallback. If a collector is blocked or unavailable the fallback handles it automatically.

In [18]:
print("=" * 60)
print("LIVE PIPELINE TEST")
print("=" * 60)

clear_cache()  # ensure we are not reading a stale cached result

engine = AggregationEngine("../Data/ventures.csv")

result = engine.run_live(
    industry="Finance Technology",
    country="United States",
    year=2015
)

print(f"\n  Status         : {result['status']}")
print(f"  Mode           : {result['mode']}")
print(f"  Input          : {result['input']}")
print(f"  Score          : {result['score']}")
print(f"  Data Quality   : {result['data_quality']} ({result['quality_label']})")

print("\n  Quality Breakdown (live / fallback / missing):")
for feature, source in result["quality_breakdown"].items():
    if source == "live":
        tag = "[OK]  live"
    elif source == "fallback":
        tag = "[~~]  fallback"
    else:
        tag = "[!!]  missing"
    print(f"    {feature:<22} -> {tag}")

print("\n  Signals:")
for k, v in result["signals"].items():
    print(f"    {k:<24} : {v}")

if result["issues"]:
    print("\n  Issues:")
    for k, v in result["issues"].items():
        print(f"    {k}: {v}")
else:
    print("\n  Issues         : none")

# Confirm mode is always 'live'
assert result["mode"] == "live", "FAIL: mode field is not 'live'"
assert 0 <= result["score"] <= 100, "FAIL: score out of 0-100 range"
assert 0 <= result["data_quality"] <= 1, "FAIL: data_quality out of 0-1 range"
print("\nAll assertions passed.")

LIVE PIPELINE TEST
[2026-02-19 19:06:08] [INFO] Cache cleared
[2026-02-19 19:06:08] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-19 19:06:08] [INFO] AggregationEngine initialized
[2026-02-19 19:06:11] [INFO] GoogleTrends: trend_slope=-0.0001
[2026-02-19 19:06:11] [INFO] GoogleTrendsCollector returned signals: ['trend_slope']
[2026-02-19 19:06:12] [INFO] WorldBank: gdp_growth=2.94555045484053
[2026-02-19 19:06:13] [INFO] WorldBank: inflation=0.118627135552451
[2026-02-19 19:06:13] [INFO] WorldBankCollector returned signals: ['gdp_growth', 'inflation']
[2026-02-19 19:06:18] [INFO] Reddit (PullPush): 'Finance Technology' 2015 -> density=2.7815
[2026-02-19 19:06:18] [INFO] RedditCollector returned signals: ['reddit_density']
[2026-02-19 19:06:25] [WARN] GDELT BQ: no results for 'Finance Technology' 2015
[2026-02-19 19:06:25] [INFO] GDELTCollector returned signals: []
[2026-02-19 19:06:25] [WARN] No exact match for (Finance Technology, United States, 2015). 

#### Fallback Test

Proves the safety net works end-to-end.

We deliberately pass an industry that **no live collector will meaningfully match** AND
a year that is outside ventures.csv's date range so the CSV fallback also finds nothing.

**Expected behaviour:**
- GoogleTrends returns `{}` (unknown industry or rate-limited)
- WorldBank **may** return live gdp_growth and inflation for United States/1995 (it covers historical data)
- RedditCollector returns near-zero density (no posts about basket weaving)
- GDELTCollector returns `{}` (no theme code mapped for this industry)
- VenturesCollector finds no exact or range match, returns `{}`
- Missing signals (trend_slope, news_sentiment, news_volume, funding) are **imputed from medians**
- Pipeline returns `status: success` with a valid numeric score — does NOT crash
- `quality_label` will be **Low or Medium** depending on how many WorldBank signals resolved
  (if WorldBank returned gdp+inflation live → Medium; if not → Low)

**Score is no longer NaN:** the expanded imputation in `compute_opportunity_score` now
covers all eight scored signals, so a missing trend_slope or funding can never poison the result.

In [19]:
print("=" * 60)
print("FALLBACK TEST  (bad input — no live data, no CSV match)")
print("=" * 60)

clear_cache()

fallback_result = engine.run_live(
    industry="Underwater Basket Weaving",  # will never match any live collector
    country="United States",
    year=1995                              # outside ventures.csv date range (2005-2018)
)

print(f"\n  Status         : {fallback_result['status']}")
print(f"  Mode           : {fallback_result['mode']}")
print(f"  Score          : {fallback_result['score']}   <- scored via median imputation")
print(f"  Data Quality   : {fallback_result['data_quality']} ({fallback_result['quality_label']})")

print("\n  Quality Breakdown:")
for feature, source in fallback_result["quality_breakdown"].items():
    if source == "live":
        tag = "[OK]  live"
    elif source == "fallback":
        tag = "[~~]  fallback"
    else:
        tag = "[!!]  missing"
    print(f"    {feature:<22} -> {tag}")

print("\n  Issues:")
if fallback_result["issues"]:
    for k, v in fallback_result["issues"].items():
        print(f"    {k}: {v}")
else:
    print("    none (medians imputed silently - score is low-confidence)")

# Confirm pipeline did not crash and still returned valid structure
assert fallback_result["status"] == "success",  "FAIL: pipeline crashed on bad input"
assert fallback_result["mode"]   == "live",      "FAIL: mode field missing"
assert 0 <= fallback_result["score"] <= 100,     "FAIL: score out of range"
assert fallback_result["quality_label"] in ("Low", "Medium"), "FAIL: expected Low or Medium quality label when most signals are fallback/imputed"

print("\nAll assertions passed. Fallback + imputation worked correctly.")

FALLBACK TEST  (bad input — no live data, no CSV match)
[2026-02-19 19:06:25] [INFO] Cache cleared
[2026-02-19 19:06:28] [ERROR] GoogleTrendsCollector failed: The request failed: Google returned a response with code 429
[2026-02-19 19:06:28] [INFO] GoogleTrendsCollector returned signals: []
[2026-02-19 19:06:29] [INFO] WorldBank: gdp_growth=2.6844307317787
[2026-02-19 19:06:29] [INFO] WorldBank: inflation=2.80541968853662
[2026-02-19 19:06:29] [INFO] WorldBankCollector returned signals: ['gdp_growth', 'inflation']
[2026-02-19 19:06:29] [WARN] RedditCollector: no keyword map for 'Underwater Basket Weaving', using name as query
[2026-02-19 19:06:30] [INFO] Reddit (PullPush): 'Underwater Basket Weaving' 1995 -> density=0.0
[2026-02-19 19:06:30] [INFO] RedditCollector returned signals: ['reddit_density']
[2026-02-19 19:06:30] [WARN] GDELTCollector: no theme code for 'Underwater Basket Weaving', skipping
[2026-02-19 19:06:30] [INFO] GDELTCollector returned signals: []
[2026-02-19 19:06:30] 

#### Mode A Demo: Historical Batch Scoring for ML Team

In [20]:
scored_df = engine.run_historical(sample_n=100)

scored_df.to_csv("historical_scored.csv", index=False)
print("Saved: historical_scored.csv")

print("\nCorrelation — pipeline_score vs ground-truth opportunity_score:")
print(scored_df[["opportunity_score", "pipeline_score"]].corr())
print()
print(scored_df[["opportunity_score", "pipeline_score", "data_quality"]].describe())

[2026-02-19 19:06:30] [INFO] run_historical: scoring 100 rows...
[2026-02-19 19:06:30] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-19 19:06:30] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-19 19:06:30] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-19 19:06:30] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallba

[2026-02-19 19:06:30] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-19 19:06:30] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-19 19:06:30] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-19 19:06:30] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-19 19:06:30] [INFO] Data quality: 0.5 (Medium) | br

---

# FastAPI Integration Guide

This section documents how the backend engineer connects this pipeline to the FastAPI layer.
The pipeline is the **data and scoring engine** — FastAPI is the **transport layer** around it.
They are kept completely separate so the pipeline can be tested and run independently.

---

## How the Integration Works

The backend engineer imports `AggregationEngine` and calls `run_live()` inside a FastAPI route.
The output dict from `run_live()` maps directly to the API response model with no transformation.

```
FastAPI POST /score   ->  AggregationEngine.run_live()        ->  JSON response
FastAPI GET  /batch   ->  AggregationEngine.run_historical()  ->  CSV or JSON
```

---

## Step-by-Step for the Backend Engineer

### 1. Install dependencies
```bash
pip install fastapi uvicorn pytrends requests
```

### 2. Recommended project structure
```
opportunity_scout/
├── data_pipeline/
│   ├── pipeline.py            <- export this notebook as Python (remove demo cells)
│   └── feature_bounds.csv     <- generated by Cell 3, must be present at runtime
├── api/
│   ├── main.py                <- FastAPI app
│   └── models.py              <- Pydantic request/response models
└── Data/
    └── ventures.csv
```

### 3. Convert the notebook to a module
Export this notebook as `pipeline.py` via File -> Download as -> Python.
Delete Cells 17-20 (tests and demos) from the exported file.
The module just needs the class definitions + `feature_bounds_df` loaded at import time.

### 4. The FastAPI app  (api/main.py)
```python
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from data_pipeline.pipeline import AggregationEngine

app    = FastAPI(title="Opportunity Scout AI", version="1.0")
engine = AggregationEngine("Data/ventures.csv")  # instantiated ONCE at startup

class ScoreRequest(BaseModel):
    industry : str
    country  : str
    year     : int

class ScoreResponse(BaseModel):
    status            : str
    mode              : str     # always 'live'
    input             : dict
    score             : float
    data_quality      : float
    quality_label     : str     # 'High' | 'Medium' | 'Low'
    quality_breakdown : dict
    signals           : dict
    issues            : dict

@app.post("/score", response_model=ScoreResponse)
def score_opportunity(request: ScoreRequest):
    try:
        return engine.run_live(
            industry=request.industry,
            country=request.country,
            year=request.year
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
def health():
    return {"status": "ok"}
```

### 5. Run locally
```bash
uvicorn api.main:app --reload
```

### 6. Example curl request
```bash
curl -X POST http://localhost:8000/score \
     -H 'Content-Type: application/json' \
     -d '{"industry": "Finance Technology", "country": "United States", "year": 2024}'
```

### 7. Example response
```json
{
  "status": "success",
  "mode": "live",
  "input": {"industry": "Finance Technology", "country": "United States", "year": 2024},
  "score": 64.32,
  "data_quality": 0.833,
  "quality_label": "High",
  "quality_breakdown": {
    "trend_slope": "live",
    "gdp_growth": "live",
    "inflation": "live",
    "reddit_density": "fallback",
    "news_volume": "live",
    "news_sentiment": "live"
  },
  "signals": { "..." },
  "issues": {}
}
```

---

## What Each Team Uses From This Pipeline

| Team              | What they call                     | What they get                                      |
|-------------------|------------------------------------|----------------------------------------------------|  
| Backend / API     | `engine.run_live()`                | Full JSON -> wrap directly in `/score` endpoint    |
| ML Engineer       | `engine.run_historical()`          | DataFrame -> `historical_scored.csv` for training  |
| Explanation/SHAP  | `result['signals']`                | Feature dict -> compute SHAP values from it        |
| Frontend          | `result['score']`, `quality_label` | Score gauge + quality badge in the UI              |
| Public API        | Same `/score` endpoint             | Externally documented version of the same JSON     |

---

## Important Notes for the Backend Engineer

- `AggregationEngine` must be instantiated **once at app startup**, not per request. Instantiating it per request would reload ventures.csv on every single call.
- The in-memory `CACHE` dict works fine in development. For production, swap it with a Redis client using the same `get_cached` / `set_cache` interface — the rest of the code does not need to change.
- The `RATE_LIMITERS` are module-level singletons and persist automatically across all requests in the same process. No extra wiring needed.
- If a live collector fails (network down, API blocked), the pipeline falls back silently and still returns HTTP 200 with a lower `data_quality` score. No 500 errors are thrown by the pipeline itself — only the FastAPI wrapper should raise HTTPException.
- The `/what-if` endpoint can be built by calling `run_live()` with a modified input dict (e.g. change the year or industry) and comparing the two scores.
- The `/compare` endpoint can be built by calling `run_live()` twice with different inputs and returning both results side by side.